## 1. Load Dataset

In [3]:
from datasets import load_dataset

ds = load_dataset("visolex/VLSP2018-ABSA-Restaurant")

print(ds)

c:\Users\thevi\Documents\Code\viet-restaurant-absa\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['Review', 'AMBIENCE#GENERAL', 'DRINKS#PRICES', 'DRINKS#QUALITY', 'DRINKS#STYLE&OPTIONS', 'FOOD#PRICES', 'FOOD#QUALITY', 'FOOD#STYLE&OPTIONS', 'LOCATION#GENERAL', 'RESTAURANT#GENERAL', 'RESTAURANT#MISCELLANEOUS', 'RESTAURANT#PRICES', 'SERVICE#GENERAL', 'type', 'dataset'],
        num_rows: 4751
    })
})


## 2. Convert into single lable

In [4]:
import pandas as pd

df = pd.DataFrame(ds["train"])

aspect_cols = [
    'AMBIENCE#GENERAL',
    'DRINKS#PRICES',
    'DRINKS#QUALITY',
    'DRINKS#STYLE&OPTIONS',
    'FOOD#PRICES',
    'FOOD#QUALITY',
    'FOOD#STYLE&OPTIONS',
    'LOCATION#GENERAL',
    'RESTAURANT#GENERAL',
    'RESTAURANT#MISCELLANEOUS',
    'RESTAURANT#PRICES',
    'SERVICE#GENERAL'
]

sentiment_map = {
    0: None,
    1: "negative",
    2: "neutral",
    3: "positive"
}

data = []

for _, row in df.iterrows():
    review = row["Review"]

    for aspect in aspect_cols:
        label = row[aspect]
        sentiment = sentiment_map.get(label)

        if sentiment:
            data.append({
                "review": review,
                "aspect": aspect,
                "sentiment": sentiment
            })

new_df = pd.DataFrame(data)

print(new_df.head())
print(f"{len(new_df)} rows")

                                              review              aspect  \
0  _ Ảnh chụp từ hôm qua, đi chơi với gia đình và...        FOOD#QUALITY   
1  _ Ảnh chụp từ hôm qua, đi chơi với gia đình và...  FOOD#STYLE&OPTIONS   
2  _Hương vị thơm ngon, ăn cay cay rất thích, nêm...    AMBIENCE#GENERAL   
3  _Hương vị thơm ngon, ăn cay cay rất thích, nêm...         FOOD#PRICES   
4  _Hương vị thơm ngon, ăn cay cay rất thích, nêm...        FOOD#QUALITY   

  sentiment  
0  positive  
1  positive  
2  negative  
3  positive  
4  negative  
15159 rows


## 3. Split into train, validation and test sets

In [5]:
import sys
import os

sys.path.append(os.path.abspath('../src'))

from utils import get_path, save_csv, split_train_val_test

train_df, val_df, test_df = split_train_val_test(
    new_df, 
    text_col="review", 
    test_size=0.2,
    val_size=0.1
)

save_csv(train_df, get_path("data", "processed", "train.csv"))
save_csv(val_df, get_path("data", "processed", "val.csv"))
save_csv(test_df, get_path("data", "processed", "test.csv"))

print(len(train_df), len(val_df), len(test_df))
print(train_df["aspect"].value_counts())

Saved to c:\Users\thevi\Documents\Code\viet-restaurant-absa\data\processed\train.csv
Saved to c:\Users\thevi\Documents\Code\viet-restaurant-absa\data\processed\val.csv
Saved to c:\Users\thevi\Documents\Code\viet-restaurant-absa\data\processed\test.csv
10564 1554 3041
aspect
FOOD#QUALITY                2993
FOOD#STYLE&OPTIONS          1860
FOOD#PRICES                 1693
RESTAURANT#GENERAL           922
AMBIENCE#GENERAL             828
SERVICE#GENERAL              807
LOCATION#GENERAL             482
RESTAURANT#MISCELLANEOUS     275
RESTAURANT#PRICES            269
DRINKS#QUALITY               163
DRINKS#PRICES                157
DRINKS#STYLE&OPTIONS         115
Name: count, dtype: int64
